# LogiCast — Baseline Model Development

## Objective
Train and evaluate machine-learning models to predict late-delivery risk.

## Target
`Late_delivery_risk`

## Models
1. Dummy Classifier
2. Logistic Regression
3. Random Forest
4. XGBoost

## Evaluation
- ROC-AUC
- PR-AUC
- Precision, Recall and F1-score
- Confusion Matrix

## Methodology
- Preserve the chronological 80/20 train-test split.
- Reuse the preprocessing pipeline from Notebook 02.
- Fit candidate models on training data only.
- Reserve the test set for final evaluation.

**Important:** This notebook does not yet deploy the model or establish production performance.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "supply_chain_cleaned.csv"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "src"
    / "models"
)

PREPROCESSOR_PATH = (
    MODEL_DIR
    / "preprocessor.joblib"
)

FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "feature_columns.txt"
)

assert PROCESSED_PATH.exists(), (
    f"Cleaned dataset not found: {PROCESSED_PATH}"
)

assert PREPROCESSOR_PATH.exists(), (
    f"Preprocessor not found: {PREPROCESSOR_PATH}"
)

assert FEATURES_PATH.exists(), (
    f"Feature list not found: {FEATURES_PATH}"
)

print("Project paths verified.")

Project paths verified.


In [4]:
df = pd.read_csv(PROCESSED_PATH)

preprocessor = joblib.load(PREPROCESSOR_PATH)

feature_cols = FEATURES_PATH.read_text().splitlines()

TARGET = "Late_delivery_risk"
ORDER_DATE = "order date (DateOrders)"

print("Dataset shape:", df.shape)
print("Feature count:", len(feature_cols))
print("Preprocessor loaded:", type(preprocessor).__name__)

display(df.head())

Dataset shape: (180519, 23)
Feature count: 17
Preprocessor loaded: ColumnTransformer


,Type,Days for shipment (scheduled),Late_delivery_risk,Category Name,Customer City,Customer Country,Customer Segment,Customer State,Department Name,Market,...,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Quantity,Order Region,Order State,Shipping Mode,Order_Month,Order_DayOfWeek,Order_Day
0,DEBIT,4,0,Sporting Goods,Caguas,Puerto Rico,Consumer,PR,Fitness,Pacific Asia,...,13.110000,0.04,327.75,1,Southeast Asia,Java Occidental,Standard Class,1,2,31
1,TRANSFER,4,1,Sporting Goods,Caguas,Puerto Rico,Consumer,PR,Fitness,Pacific Asia,...,16.389999,0.05,327.75,1,South Asia,Rajastán,Standard Class,1,5,13
2,CASH,4,0,Sporting Goods,San Jose,EE. UU.,Consumer,CA,Fitness,Pacific Asia,...,18.030001,0.06,327.75,1,South Asia,Rajastán,Standard Class,1,5,13
3,DEBIT,4,0,Sporting Goods,Los Angeles,EE. UU.,Home Office,CA,Fitness,Pacific Asia,...,22.940001,0.07,327.75,1,Oceania,Queensland,Standard Class,1,5,13
4,PAYMENT,4,0,Sporting Goods,Caguas,Puerto Rico,Corporate,PR,Fitness,Pacific Asia,...,29.500000,0.09,327.75,1,Oceania,Queensland,Standard Class,1,5,13


In [5]:
df[ORDER_DATE] = pd.to_datetime(
    df[ORDER_DATE],
    errors="coerce"
)

df = (
    df
    .dropna(subset=[TARGET, ORDER_DATE])
    .sort_values(ORDER_DATE)
    .reset_index(drop=True)
)

df[TARGET] = df[TARGET].astype(int)

split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining period:")
print(train_df[ORDER_DATE].min(), "to", train_df[ORDER_DATE].max())

print("\nTesting period:")
print(test_df[ORDER_DATE].min(), "to", test_df[ORDER_DATE].max())

Training rows: 144415
Testing rows: 36104

Training period:
2015-01-01 00:00:00 to 2017-04-22 15:17:00

Testing period:
2017-04-22 15:17:00 to 2018-01-31 23:38:00


In [6]:
X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET].copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).sort_index())

X_train: (144415, 17)
X_test : (36104, 17)

Training target distribution:
Late_delivery_risk
0    0.451276
1    0.548724
Name: proportion, dtype: float64

Testing target distribution:
Late_delivery_risk
0    0.45344
1    0.54656
Name: proportion, dtype: float64


In [7]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape :", X_test_processed.shape)

Processed training shape: (144415, 1327)
Processed testing shape : (36104, 1327)


In [8]:
dummy_model = DummyClassifier(
    strategy="most_frequent"
)

dummy_model.fit(
    X_train_processed,
    y_train
)

dummy_predictions = dummy_model.predict(
    X_test_processed
)

print(
    classification_report(
        y_test,
        dummy_predictions,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     16371
           1       0.55      1.00      0.71     19733

    accuracy                           0.55     36104
   macro avg       0.27      0.50      0.35     36104
weighted avg       0.30      0.55      0.39     36104

